# Exerkine panel to FASTA
Fetches the **human RefSeq CDS (nucleotide)** from NCBI and the **reviewed protein** from UniProt for every protein-coding exerkine in your tiered panel, and writes FASTA files plus a manifest. Metabolites and the engineered chimera have no gene sequence and are listed as skipped.

Outputs: `exerkines_cds_human.fasta` (RNA alphabet if `AS_RNA`), `exerkines_protein_human.fasta`, `exerkines_manifest.csv`.

In [1]:
!pip -q install biopython pandas requests

In [2]:
import io, time, requests, pandas as pd
from Bio import SeqIO, Entrez
from Bio.Seq import Seq

Entrez.email = "you@example.com"   # <-- set your email (NCBI requirement)
# Entrez.api_key = "..."           # optional, raises the rate limit
AS_RNA = False                     # True writes U instead of T in the CDS FASTA
ENZYME_PROXIES = False             # True also fetches enzymes tied to the metabolites (NOT the molecules themselves)

# (name, tier, gene, note)   gene=None -> no gene-encoded sequence
PANEL = [
 ("IL-6",1,"IL6",""), ("IGF-1",1,"IGF1",""), ("VEGF",1,"VEGFA",""),
 ("BHB",1,None,"metabolite"),
 ("NRG1",2,"NRG1","many isoforms"), ("GDNF",2,"GDNF",""), ("Follistatin",2,"FST",""),
 ("Lac-Phe",2,None,"metabolite"), ("GDF15",2,"GDF15",""), ("BDNF",2,"BDNF",""),
 ("IC7Fc",2,None,"engineered IL-6/CNTF chimera; use the published construct sequence"),
 ("Lactate",2,None,"metabolite"), ("Klotho",2,"KL",""),
 ("Irisin",3,"FNDC5","irisin is a cleavage product; full precursor plus a derived irisin record"),
 ("FSTL1",3,"FSTL1",""), ("ELABELA",3,"APELA",""), ("Fractalkine",3,"CX3CL1",""),
 ("Omentin",3,"ITLN1",""), ("Apelin",3,"APLN",""), ("Adiponectin",3,"ADIPOQ",""),
 ("Decorin",3,"DCN",""), ("KYNA",3,None,"metabolite"), ("Clusterin",3,"CLU",""),
 ("HSP72",3,"HSPA1A","HSPA1A/HSPA1B encode the same protein"), ("BAIBA",3,None,"metabolite"),
 ("Myonectin",4,"ERFE","CTRP15 / erythroferrone"), ("Lubricin",4,"PRG4",""), ("PF4",4,"PF4",""),
 ("TGF-b2",4,"TGFB2",""), ("VIP",4,"VIP",""), ("CO",4,None,"gas"),
 ("Cathepsin B",4,"CTSB",""), ("GPLD1",4,"GPLD1",""), ("FGF2",4,"FGF2",""),
 ("SeP",4,"SELENOP","contains in-frame TGA (selenocysteine); protein shows U"),
]
PROXIES = {"Lac-Phe":"CNDP2", "CO":"HMOX1", "Lactate":"LDHA", "BHB":"HMGCS2", "KYNA":"AADAT"}
if ENZYME_PROXIES:
    PANEL += [(f"{n} enzyme",t,PROXIES[n],"enzyme proxy, not the molecule") for n,t,g,_ in PANEL if g is None and n in PROXIES]

## Fetch

In [3]:
def cds_from_record(rec, gene):
    for f in rec.features:
        if f.type == "CDS" and f.qualifiers.get("gene", [None])[0] == gene:
            return dict(acc=rec.id, cds=str(f.extract(rec.seq)).upper(),
                        ncbi_protein=f.qualifiers.get("translation", [""])[0],
                        product=f.qualifiers.get("product", [""])[0])
    return None

def ncbi_cds(gene, organism="Homo sapiens", n=5):
    term = f"{gene}[Gene Name] AND {organism}[Organism] AND srcdb_refseq_known[Properties] AND biomol_mrna[Properties]"
    ids = Entrez.read(Entrez.esearch(db="nuccore", term=term, retmax=n))["IdList"]; time.sleep(0.4)
    best = None
    for i in ids:                      # several transcript variants: keep the longest CDS
        rec = SeqIO.read(Entrez.efetch(db="nuccore", id=i, rettype="gb", retmode="text"), "genbank"); time.sleep(0.4)
        r = cds_from_record(rec, gene)
        if r and (best is None or len(r["cds"]) > len(best["cds"])): best = r
    return best

def uniprot_protein(gene, taxon=9606):
    q = f"(gene_exact:{gene}) AND (organism_id:{taxon}) AND (reviewed:true)"
    r = requests.get("https://rest.uniprot.org/uniprotkb/search",
                     params={"query": q, "format": "fasta", "size": 1}, timeout=30)
    recs = list(SeqIO.parse(io.StringIO(r.text), "fasta"))
    return (recs[0].id.split("|")[1], str(recs[0].seq)) if recs else (None, None)

records, rows = [], []
for name, tier, gene, note in PANEL:
    if gene is None:
        rows.append(dict(name=name, tier=tier, gene=None, status="skipped", note=note)); continue
    c = ncbi_cds(gene); up_acc, up_seq = uniprot_protein(gene)
    if not c:
        rows.append(dict(name=name, tier=tier, gene=gene, status="no RefSeq CDS", note=note)); continue
    tr = str(Seq(c["cds"]).translate(to_stop=True))
    rows.append(dict(name=name, tier=tier, gene=gene, status="ok", refseq=c["acc"], uniprot=up_acc,
                     cds_nt=len(c["cds"]), protein_aa=len(up_seq or ""),
                     translates_to_ncbi_protein=(tr == c["ncbi_protein"].rstrip("*")),
                     translates_to_uniprot=(tr == up_seq) if up_seq else None, note=note))
    records.append((name, tier, gene, c, up_acc, up_seq))

    if gene == "FNDC5":                # irisin = precursor residues 32-143 (verify against your source)
        records.append((name + "_irisin", tier, gene + "_irisin",
                        dict(acc=c["acc"], cds=c["cds"][93:429]), up_acc, (up_seq or "")[31:143]))
manifest = pd.DataFrame(rows); manifest

,name,tier,gene,status,refseq,uniprot,cds_nt,protein_aa,translates_to_ncbi_protein,translates_to_uniprot,note
0,IL-6,1,IL6,ok,NM_000600.5,P05231,639.0,212.0,True,True,
1,IGF-1,1,IGF1,ok,NM_001414005.1,P05019,588.0,195.0,True,True,
2,VEGF,1,VEGFA,ok,NM_001171623.2,P15692,699.0,395.0,True,False,
3,BHB,1,NaN,skipped,NaN,NaN,NaN,NaN,NaN,NaN,metabolite
4,NRG1,2,NRG1,ok,NM_013962.3,Q02297,1269.0,640.0,True,False,many isoforms
5,GDNF,2,GDNF,ok,NM_001190468.1,P39905,687.0,211.0,True,False,
6,Follistatin,2,FST,ok,NM_013409.3,P19883,1035.0,344.0,True,True,
7,Lac-Phe,2,NaN,skipped,NaN,NaN,NaN,NaN,NaN,NaN,metabolite
8,GDF15,2,GDF15,ok,NM_004864.4,Q99988,927.0,308.0,True,True,
9,BDNF,2,BDNF,ok,NM_001143810.2,P23560,990.0,247.0,True,False,


## Write FASTA

In [4]:
def wrap(s, n=70): return "\n".join(s[i:i+n] for i in range(0, len(s), n))
with open("exerkines_cds_human.fasta", "w") as fn, open("exerkines_protein_human.fasta", "w") as fp:
    for name, tier, gene, c, up_acc, up_seq in records:
        nt = c["cds"].replace("T", "U") if AS_RNA else c["cds"]
        fn.write(f">{gene}|{name}|tier{tier}|{c['acc']}|Homo_sapiens\n{wrap(nt)}\n")
        if up_seq: fp.write(f">{gene}|{name}|tier{tier}|{up_acc}|Homo_sapiens\n{wrap(up_seq)}\n")
manifest.to_csv("exerkines_manifest.csv", index=False)
print(len(records), "records written;", (manifest.status != "ok").sum(), "skipped or missing")

27 records written; 9 skipped or missing


**Notes**
- CDS records are the *precursor* (prepro) coding sequence from the longest RefSeq variant, not the mature secreted peptide. Check `translates_to_*` in the manifest; a False usually means isoform differences or, for SeP, the selenocysteine codon.
- Irisin slice boundaries (precursor 32-143) are my assumption; verify against your reference.
- NRG1, IGF1 and VEGFA have multiple isoforms; choose the isoform you need if the longest is not the right one.